### **Broadcasting 1**

In [31]:
def standardize_rows(data, mean, std):
    import numpy as np

    n = len(data)
    d = len(mean)

    output = np.zeros((n, d))

    for i in range(n):
        for j in range(d):
            output[i][j] = (data[i][j] - mean[j]) / std[j]

    return output

### **Broadcasting 2**

In [32]:
def outer(x, y):
    import numpy as np

    output = np.zeros((len(x), len(y)))

    for i in range(len(x)):
        for j in range(len(y)):
            output[i][j] = x[i] * y[j]
    return output

In [33]:
import numpy as np

x = np.array([1,2])
y = np.array([3,4,5])

outer(x, y)

array([[ 3.,  4.,  5.],
       [ 6.,  8., 10.]])

### **Broadcasting 3**

In [ ]:
def distmat_1d(x, y):
    return abs(x.reshape(-1, 1) - y.reshape(1, -1))

In [ ]:
import numpy as np

x = np.array([1,2])
y = np.array([3,0.5,1])

distmat_1d(x, y)

array([[2. , 0.5, 0. ],
       [1. , 1.5, 1. ]])

### **High Performance Haversine 1**

We consider the following python program:
```python
import sys
import numpy as np

def distance_matrix(p1, p2):
    p1, p2 = np.radians(p1), np.radians(p2)

    D = np.empty((len(p1), len(p2)))
    for i in range(len(p1)):
        for j in range(len(p2)):
            dsin2 = np.sin(0.5 * (p1[i] - p2[j])) ** 2
            cosprod = np.cos(p1[i, 0]) * np.cos(p2[j, 0])
            a = dsin2[0] + cosprod * dsin2[1]
            D[i, j] = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    D *= 6371  # Earth radius in km
    return D


def load_points(fname):
    data = np.loadtxt(fname, delimiter=',', skiprows=1, usecols=(1, 2))
    return data


def distance_stats(D):
    # Extract upper triangular part to avoid duplicate entries
    assert D.shape[0] == D.shape[1], 'D must be square'
    idx = np.triu_indices(D.shape[0], k=1)
    distances = D[idx]
    return {
        'mean': float(distances.mean()),
        'std': float(distances.std()),
        'max': float(distances.max()),
        'min': float(distances.min()),
    }


fname = sys.argv[1]
points = load_points(fname)
D = distance_matrix(points, points)
stats = distance_stats(D)
print(stats)
```

Make a job script that runs the above program on the hpc queue. Request a single core and specify a CPU model so the results are repeatable. For the Autolab submission, assume that the input is always a file with path input.csv.

### **High Performance Haversine 2**

Just added:
```python
python3 -m cProfile -s tottime haversine.py input.csv
```
To the .sh file.

### **High Performance Haversine 3**

See the distance_matrix function.

In [ ]:
import sys
import numpy as np

def distance_matrix(p1, p2):
    p1, p2 = np.radians(p1), np.radians(p2)

    D = np.empty((len(p1), len(p2)))

    for i in range(len(p1)):
        dsin2 = np.sin(0.5 * (p1[i]- p2)) ** 2
        cosprod = np.cos(p1[i,0]) * np.cos(p2[:, 0])
        a = dsin2[:, 0] + cosprod * dsin2[:, 1]
        D[i] = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    D *= 6371  # Earth radius in km
    return D

def load_points(fname):
    data = np.loadtxt(fname, delimiter=',', skiprows=1, usecols=(1, 2))
    return data


def distance_stats(D):
    # Extract upper triangular part to avoid duplicate entries
    assert D.shape[0] == D.shape[1], 'D must be square'
    idx = np.triu_indices(D.shape[0], k=1)
    distances = D[idx]
    return {
        'mean': float(distances.mean()),
        'std': float(distances.std()),
        'max': float(distances.max()),
        'min': float(distances.min()),
    }

fname = sys.argv[1]
points = load_points(fname)
D = distance_matrix(points, points)
stats = distance_stats(D)
print(stats)

### **High Performance Haversine 4**

See HPC output.

### **High Performance Haversine 5**

Just added `@profile` to the distance_matrix function, and added this at the end of the .sh file:
```bash
cd $LS_SUBCWD

echo "CPU model on this node:"
grep "model name" /proc/cpuinfo | head -n 1

echo "Running Python script now!"

kernprof -l haversine.py /dtu/projects/02613_2025/data/locations/locations_5000.csv

python3 -m line_profiler haversine.py.lprof
```
That is it :0.

### **High Performance Haversine 6**

First off, `np.cos(p2[:,0])` was computed in every iteration of the loop so we moved it out, since it did not depend on `i`. Second, we did this instead `D[i] = 2 * np.arcsin(np.sqrt(a))` via the formula.

### **High Performance Haversine 7**

We rewrite below.

In [ ]:
def distance_matrix(p1, p2):
    p1, p2 = np.radians(p1), np.radians(p2)

    D = np.empty((len(p1), len(p2)))

    dsin2 = np.sin(0.5 * (p1[:, np.newaxis] - p2)) ** 2
    
    cosprod = np.cos(p1[:, 0])[:, np.newaxis] * np.cos(p2[:, 0])

    a = dsin2[:, :, 0] + cosprod * dsin2[:, :, 1]

    D = 2 * np.arcsin(np.sqrt(a))
    
    D *= 6371  # Earth radius in km
    return D    

### **High Performance Haversine 8**

;(